# Notebook 11 — Biomarker-Integrated Model Comparison

## Objective
Compare the reference **clinical + DaTSCAN** feature set with the **clinical + DaTSCAN + biomarker** feature set for predicting rapid Parkinson’s disease motor progression.

## Scientific Background
Notebook 10 identified baseline/screening biomarker features suitable for integration with the existing clinical + DaTSCAN feature matrix. This notebook evaluates whether adding biomarkers improves internal validation performance.

## Dataset Verification
This notebook uses only processed outputs from:
- Notebook 08: clinical + DaTSCAN preprocessing.
- Notebook 10: clinical + DaTSCAN + biomarker preprocessing.

No raw PPMI participant-level data are exported publicly.

In [ ]:
# ============================================================
# 01. Mount Google Drive
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# 02. Imports and project paths
# ============================================================

from pathlib import Path
import os
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score, confusion_matrix,
    roc_curve, precision_recall_curve, brier_score_loss
)
from sklearn.inspection import permutation_importance
import joblib

RANDOM_STATE = 42
TARGET_COL = "rapid_progression_q75"
ID_COL = "PATNO"

PROJECT_DIR = Path("/content/drive/MyDrive/PPMI_PD_Progression")

NB08_DIR = PROJECT_DIR / "outputs" / "notebook_08_multimodal_preprocessing"
NB10_DIR = PROJECT_DIR / "outputs" / "notebook_10_biomarker_integration"
OUT_DIR = PROJECT_DIR / "outputs" / "notebook_11_biomarker_model_comparison"

OUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("Notebook 08 directory:", NB08_DIR, "| exists:", NB08_DIR.exists())
print("Notebook 10 directory:", NB10_DIR, "| exists:", NB10_DIR.exists())
print("Notebook 11 output directory:", OUT_DIR)

## Code — Input file verification

In [ ]:
# ============================================================
# 03. Verify required input files
# ============================================================

required_files = {
    "nb08_train_processed": NB08_DIR / "08_multimodal_train_processed_matrix.csv",
    "nb08_test_processed": NB08_DIR / "09_multimodal_test_processed_matrix.csv",
    "nb10_train_processed": NB10_DIR / "12_biomarker_integrated_train_processed_matrix.csv",
    "nb10_test_processed": NB10_DIR / "13_biomarker_integrated_test_processed_matrix.csv",
}

file_check = []
for label, path in required_files.items():
    file_check.append({
        "input_label": label,
        "path": str(path),
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else np.nan,
    })

file_check_df = pd.DataFrame(file_check)
display(file_check_df)
file_check_df.to_csv(OUT_DIR / "01_input_file_check.csv", index=False)

missing = file_check_df.loc[~file_check_df["exists"], "path"].tolist()
if missing:
    raise FileNotFoundError("Missing required input files:\n" + "\n".join(missing))

In [ ]:
# ============================================================
# 04. Load feature matrices
# ============================================================

feature_sets = {
    "clinical_plus_datscan": {
        "train_path": required_files["nb08_train_processed"],
        "test_path": required_files["nb08_test_processed"],
    },
    "clinical_plus_datscan_plus_biomarkers": {
        "train_path": required_files["nb10_train_processed"],
        "test_path": required_files["nb10_test_processed"],
    },
}

loaded = {}
shape_rows = []

for fs_name, paths in feature_sets.items():
    train_df = pd.read_csv(paths["train_path"])
    test_df = pd.read_csv(paths["test_path"])

    train_df[ID_COL] = train_df[ID_COL].astype(str)
    test_df[ID_COL] = test_df[ID_COL].astype(str)

    loaded[fs_name] = {"train": train_df, "test": test_df}

    predictor_cols = [c for c in train_df.columns if c not in [ID_COL, TARGET_COL]]

    shape_rows.append({
        "feature_set": fs_name,
        "train_n": train_df.shape[0],
        "test_n": test_df.shape[0],
        "processed_predictor_count": len(predictor_cols),
        "train_positive_n": int(train_df[TARGET_COL].sum()),
        "test_positive_n": int(test_df[TARGET_COL].sum()),
        "train_positive_pct": round(100 * train_df[TARGET_COL].mean(), 2),
        "test_positive_pct": round(100 * test_df[TARGET_COL].mean(), 2),
        "train_missing_total": int(train_df[predictor_cols].isna().sum().sum()),
        "test_missing_total": int(test_df[predictor_cols].isna().sum().sum()),
    })

shape_summary = pd.DataFrame(shape_rows)
display(shape_summary)
shape_summary.to_csv(OUT_DIR / "02_feature_set_shapes.csv", index=False)

# Verify the same train/test participant IDs are preserved.
train_ids_ref = set(loaded["clinical_plus_datscan"]["train"][ID_COL])
test_ids_ref = set(loaded["clinical_plus_datscan"]["test"][ID_COL])

for fs_name in loaded:
    assert set(loaded[fs_name]["train"][ID_COL]) == train_ids_ref, f"Train IDs differ for {fs_name}"
    assert set(loaded[fs_name]["test"][ID_COL]) == test_ids_ref, f"Test IDs differ for {fs_name}"

print("Train/test participant IDs are preserved across feature sets.")

## Code — Model definitions and evaluation functions

In [ ]:
# ============================================================
# 05. Model definitions
# ============================================================

models = {
    "Dummy_most_frequent": DummyClassifier(strategy="most_frequent"),
    "Logistic_L2_balanced": LogisticRegression(
        penalty="l2",
        solver="liblinear",
        class_weight="balanced",
        max_iter=5000,
        random_state=RANDOM_STATE
    ),
    "RandomForest_balanced": RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=5,
        max_features="sqrt",
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "GradientBoosting": GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=2,
        random_state=RANDOM_STATE
    ),
    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_iter=200,
        learning_rate=0.05,
        max_leaf_nodes=15,
        l2_regularization=0.1,
        random_state=RANDOM_STATE
    ),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print("Models:", list(models.keys()))

In [ ]:
# ============================================================
# 06. Evaluation helper functions
# ============================================================

def get_proba(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        scores = model.decision_function(X)
        return 1 / (1 + np.exp(-scores))
    preds = model.predict(X)
    return preds.astype(float)


def safe_auc(y_true, y_score, kind="roc"):
    try:
        if len(np.unique(y_true)) < 2:
            return np.nan
        if kind == "roc":
            return roc_auc_score(y_true, y_score)
        if kind == "pr":
            return average_precision_score(y_true, y_score)
    except Exception:
        return np.nan


def classify_with_threshold(y_score, threshold):
    return (np.asarray(y_score) >= threshold).astype(int)


def metric_dict(y_true, y_score, threshold=0.5):
    y_pred = classify_with_threshold(y_score, threshold)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    return {
        "roc_auc": safe_auc(y_true, y_score, "roc"),
        "pr_auc": safe_auc(y_true, y_score, "pr"),
        "threshold": float(threshold),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) > 0 else np.nan,
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "brier_score": brier_score_loss(y_true, y_score),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def choose_threshold_by_balanced_accuracy(y_true, y_score):
    thresholds = np.round(np.linspace(0.05, 0.95, 91), 3)
    rows = []
    for t in thresholds:
        m = metric_dict(y_true, y_score, threshold=t)
        rows.append({"threshold": t, "balanced_accuracy": m["balanced_accuracy"], "sensitivity": m["sensitivity"], "specificity": m["specificity"]})
    tab = pd.DataFrame(rows).sort_values(["balanced_accuracy", "sensitivity"], ascending=False)
    return float(tab.iloc[0]["threshold"]), tab


def evaluate_feature_set(fs_name, train_df, test_df, predictor_columns=None):
    if predictor_columns is None:
        predictor_columns = [c for c in train_df.columns if c not in [ID_COL, TARGET_COL]]

    X_train = train_df[predictor_columns].copy()
    y_train = train_df[TARGET_COL].astype(int).copy()
    X_test = test_df[predictor_columns].copy()
    y_test = test_df[TARGET_COL].astype(int).copy()

    cv_rows = []
    test_rows = []
    fitted_models = {}

    for model_name, model in models.items():
        print(f"Running {fs_name} | {model_name}")

        try:
            if model_name.startswith("Dummy"):
                cv_probs = np.repeat(y_train.mean(), len(y_train))
            else:
                cv_probs = cross_val_predict(model, X_train, y_train, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]

            selected_threshold, threshold_tab = choose_threshold_by_balanced_accuracy(y_train, cv_probs)

            cv_m = metric_dict(y_train, cv_probs, threshold=selected_threshold)
            cv_m.update({
                "feature_set": fs_name,
                "model": model_name,
                "n_train": len(y_train),
                "n_predictors": len(predictor_columns),
                "evaluation": "5fold_cv_oof",
            })
            cv_rows.append(cv_m)

            fitted_model = model.fit(X_train, y_train)
            fitted_models[model_name] = fitted_model

            test_probs = get_proba(fitted_model, X_test)
            test_m = metric_dict(y_test, test_probs, threshold=selected_threshold)
            test_m.update({
                "feature_set": fs_name,
                "model": model_name,
                "n_test": len(y_test),
                "n_predictors": len(predictor_columns),
                "evaluation": "heldout_test",
            })
            test_rows.append(test_m)

        except Exception as e:
            print(f"WARNING: {fs_name} | {model_name} failed: {e}")

    return pd.DataFrame(cv_rows), pd.DataFrame(test_rows), fitted_models

## Code — Primary clinical + DaTSCAN vs biomarker-integrated comparison

In [ ]:
# ============================================================
# 07. Run primary model comparison
# ============================================================

all_cv = []
all_test = []
all_fitted = {}

for fs_name, data in loaded.items():
    train_df = data["train"]
    test_df = data["test"]

    predictor_cols = [c for c in train_df.columns if c not in [ID_COL, TARGET_COL]]

    cv_df, test_df_perf, fitted_models = evaluate_feature_set(
        fs_name=fs_name,
        train_df=train_df,
        test_df=test_df,
        predictor_columns=predictor_cols
    )

    all_cv.append(cv_df)
    all_test.append(test_df_perf)
    all_fitted[fs_name] = fitted_models

cv_performance = pd.concat(all_cv, ignore_index=True)
test_performance = pd.concat(all_test, ignore_index=True)

# Sort by CV ROC-AUC primarily, not test performance.
cv_performance = cv_performance.sort_values(["roc_auc", "pr_auc"], ascending=False)
test_performance = test_performance.sort_values(["feature_set", "roc_auc", "pr_auc"], ascending=[True, False, False])

cv_performance.to_csv(OUT_DIR / "03_cv_performance_biomarker_comparison.csv", index=False)
test_performance.to_csv(OUT_DIR / "04_test_performance_biomarker_comparison.csv", index=False)

display(cv_performance)
display(test_performance)

In [ ]:
# ============================================================
# 08. Best model by feature set and incremental biomarker performance
# ============================================================

# Choose best model per feature set using CV ROC-AUC, then report held-out test performance.
best_by_feature_set = (
    cv_performance
    .sort_values(["feature_set", "roc_auc", "pr_auc"], ascending=[True, False, False])
    .groupby("feature_set", as_index=False)
    .head(1)[["feature_set", "model", "roc_auc", "pr_auc", "balanced_accuracy", "sensitivity", "specificity", "f1"]]
    .rename(columns={
        "roc_auc": "cv_roc_auc",
        "pr_auc": "cv_pr_auc",
        "balanced_accuracy": "cv_balanced_accuracy",
        "sensitivity": "cv_sensitivity",
        "specificity": "cv_specificity",
        "f1": "cv_f1",
    })
)

best_test = best_by_feature_set[["feature_set", "model"]].merge(
    test_performance,
    on=["feature_set", "model"],
    how="left"
)

best_test.to_csv(OUT_DIR / "05_best_model_by_feature_set.csv", index=False)
display(best_test)

ref_name = "clinical_plus_datscan"
bio_name = "clinical_plus_datscan_plus_biomarkers"

if set([ref_name, bio_name]).issubset(set(best_test["feature_set"])):
    ref = best_test[best_test["feature_set"] == ref_name].iloc[0]
    bio = best_test[best_test["feature_set"] == bio_name].iloc[0]

    incremental = pd.DataFrame([{
        "reference_feature_set": ref_name,
        "biomarker_feature_set": bio_name,
        "reference_model": ref["model"],
        "biomarker_model": bio["model"],
        "delta_test_roc_auc": bio["roc_auc"] - ref["roc_auc"],
        "delta_test_pr_auc": bio["pr_auc"] - ref["pr_auc"],
        "delta_test_balanced_accuracy": bio["balanced_accuracy"] - ref["balanced_accuracy"],
        "delta_test_sensitivity": bio["sensitivity"] - ref["sensitivity"],
        "delta_test_specificity": bio["specificity"] - ref["specificity"],
        "delta_test_f1": bio["f1"] - ref["f1"],
    }])
else:
    incremental = pd.DataFrame()

incremental.to_csv(OUT_DIR / "06_biomarker_incremental_performance_summary.csv", index=False)
display(incremental)

In [ ]:
# ============================================================
# 09. Threshold table for selected best overall model
# ============================================================

# Select overall best model based on CV ROC-AUC.
selected = cv_performance.iloc[0]
selected_fs = selected["feature_set"]
selected_model_name = selected["model"]

selected_train = loaded[selected_fs]["train"]
selected_test = loaded[selected_fs]["test"]
selected_predictors = [c for c in selected_train.columns if c not in [ID_COL, TARGET_COL]]

X_train_sel = selected_train[selected_predictors]
y_train_sel = selected_train[TARGET_COL].astype(int)
X_test_sel = selected_test[selected_predictors]
y_test_sel = selected_test[TARGET_COL].astype(int)

selected_model = models[selected_model_name].fit(X_train_sel, y_train_sel)
test_probs_sel = get_proba(selected_model, X_test_sel)

threshold_rows = []
for t in np.round(np.linspace(0.05, 0.95, 91), 3):
    row = metric_dict(y_test_sel, test_probs_sel, threshold=t)
    row.update({
        "feature_set": selected_fs,
        "model": selected_model_name,
    })
    threshold_rows.append(row)

threshold_table = pd.DataFrame(threshold_rows)
threshold_table = threshold_table.sort_values(["balanced_accuracy", "sensitivity"], ascending=False)

threshold_table.to_csv(OUT_DIR / "07_threshold_table_selected_best_model_test.csv", index=False)
display(threshold_table.head(20))

print("Selected best overall model:", selected_fs, "|", selected_model_name)

In [ ]:
# ============================================================
# 10. Permutation importance for selected best model on held-out test set
# ============================================================

try:
    perm = permutation_importance(
        selected_model,
        X_test_sel,
        y_test_sel,
        n_repeats=20,
        scoring="roc_auc",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    importance_df = pd.DataFrame({
        "feature": selected_predictors,
        "importance_mean": perm.importances_mean,
        "importance_sd": perm.importances_std,
    }).sort_values("importance_mean", ascending=False)

except Exception as e:
    print("Permutation importance failed:", e)
    importance_df = pd.DataFrame({
        "feature": selected_predictors,
        "importance_mean": np.nan,
        "importance_sd": np.nan,
    })

importance_df.to_csv(OUT_DIR / "08_permutation_importance_selected_best_model_test.csv", index=False)
display(importance_df.head(30))

## Code — Sensitivity analysis without baseline_NP3TOT

In [ ]:
# ============================================================
# 11. Sensitivity analysis without baseline_NP3TOT
# ============================================================

sensitivity_rows = []

for fs_name, data in loaded.items():
    train_df = data["train"]
    test_df = data["test"]

    predictor_cols = [
        c for c in train_df.columns
        if c not in [ID_COL, TARGET_COL] and "baseline_NP3TOT" not in c
    ]

    cv_df, test_df_perf, _ = evaluate_feature_set(
        fs_name=fs_name + "_no_baseline_NP3TOT",
        train_df=train_df,
        test_df=test_df,
        predictor_columns=predictor_cols
    )

    sensitivity_rows.append(test_df_perf)

sensitivity_test = pd.concat(sensitivity_rows, ignore_index=True)
sensitivity_test = sensitivity_test.sort_values(["feature_set", "roc_auc", "pr_auc"], ascending=[True, False, False])

sensitivity_test.to_csv(OUT_DIR / "09_sensitivity_test_performance_no_baseline_NP3TOT.csv", index=False)
display(sensitivity_test)

## Scientific Interpretation
The key interpretation is whether biomarker-integrated models improve ROC-AUC, PR-AUC, balanced accuracy, sensitivity, and F1-score compared with the clinical + DaTSCAN reference model.

A small ROC-AUC increase without PR-AUC or balanced-accuracy improvement should be treated cautiously.

In [ ]:
# ============================================================
# 12. Quality Control Checklist
# ============================================================

qc_rows = []

def add_qc(item, status, details):
    qc_rows.append({"qc_item": item, "status": status, "details": details})

add_qc(
    "Notebook 08 processed matrices loaded",
    "PASS" if "clinical_plus_datscan" in loaded else "FAIL",
    f"clinical_plus_datscan train={loaded.get('clinical_plus_datscan', {}).get('train', pd.DataFrame()).shape}"
)

add_qc(
    "Notebook 10 biomarker-integrated matrices loaded",
    "PASS" if "clinical_plus_datscan_plus_biomarkers" in loaded else "FAIL",
    f"biomarker train={loaded.get('clinical_plus_datscan_plus_biomarkers', {}).get('train', pd.DataFrame()).shape}"
)

add_qc(
    "Train/test split preserved across feature sets",
    "PASS",
    "Participant IDs checked by assertion before model training."
)

add_qc(
    "No missing values in processed predictors",
    "PASS" if shape_summary["train_missing_total"].sum() == 0 and shape_summary["test_missing_total"].sum() == 0 else "FAIL",
    f"train_missing_total={shape_summary['train_missing_total'].sum()}; test_missing_total={shape_summary['test_missing_total'].sum()}"
)

add_qc(
    "Cross-validation performed on training set only",
    "PASS",
    "5-fold stratified OOF predictions used for CV performance."
)

add_qc(
    "Threshold selected using training CV only",
    "PASS",
    "Held-out test set not used for threshold selection."
)

add_qc(
    "Held-out test set evaluated once per model",
    "PASS",
    f"test_n={shape_summary['test_n'].iloc[0]}"
)

add_qc(
    "Sensitivity analysis without baseline_NP3TOT completed",
    "PASS" if sensitivity_test.shape[0] > 0 else "FAIL",
    f"sensitivity_rows={sensitivity_test.shape[0]}"
)

qc = pd.DataFrame(qc_rows)
qc.to_csv(OUT_DIR / "10_quality_control_checklist.csv", index=False)
display(qc)

if (qc["status"] == "FAIL").any():
    raise RuntimeError("One or more QC checks failed. Review 10_quality_control_checklist.csv before proceeding.")

## Expected Output
This notebook saves:
- model performance comparison,
- incremental biomarker contribution summary,
- threshold table,
- permutation importance,
- sensitivity analysis without baseline_NP3TOT,
- quality-control checklist.

In [ ]:
# ============================================================
# 13. Summary report
# ============================================================

best_row = cv_performance.iloc[0]
summary_lines = [
    "Notebook 11 — Biomarker-Integrated Model Comparison",
    "=" * 72,
    f"Project folder: {PROJECT_DIR}",
    f"Output folder: {OUT_DIR}",
    "",
    "Feature sets compared:",
]

for _, row in shape_summary.iterrows():
    summary_lines.append(
        f"- {row['feature_set']}: train={row['train_n']}, test={row['test_n']}, processed_predictors={row['processed_predictor_count']}"
    )

summary_lines += [
    "",
    "Best model by CV ROC-AUC:",
    f"- Feature set: {best_row['feature_set']}",
    f"- Model: {best_row['model']}",
    f"- CV ROC-AUC: {best_row['roc_auc']:.4f}",
    f"- CV PR-AUC: {best_row['pr_auc']:.4f}",
    "",
    "Incremental biomarker performance summary:",
]

if incremental.shape[0] > 0:
    inc = incremental.iloc[0]
    summary_lines += [
        f"- Delta test ROC-AUC: {inc['delta_test_roc_auc']:.4f}",
        f"- Delta test PR-AUC: {inc['delta_test_pr_auc']:.4f}",
        f"- Delta test balanced accuracy: {inc['delta_test_balanced_accuracy']:.4f}",
        f"- Delta test sensitivity: {inc['delta_test_sensitivity']:.4f}",
        f"- Delta test specificity: {inc['delta_test_specificity']:.4f}",
        f"- Delta test F1: {inc['delta_test_f1']:.4f}",
    ]
else:
    summary_lines.append("- Incremental comparison could not be calculated.")

summary_lines += [
    "",
    "Quality control:",
    f"- QC PASS count: {(qc['status'] == 'PASS').sum()}",
    f"- QC FAIL count: {(qc['status'] == 'FAIL').sum()}",
    "",
    "Decision point:",
    "Review whether biomarkers improve clinically relevant metrics before deciding whether to keep them in the final model.",
]

summary_text = "\n".join(summary_lines)
print(summary_text)

with open(OUT_DIR / "11_notebook_11_summary_report.txt", "w", encoding="utf-8") as f:
    f.write(summary_text)